In [2]:
import pandas as pd, numpy as np
df = pd.read_csv("../data/Zomato_Restaurant_Dataset.csv", encoding="utf-8-sig")
print(df.shape)
df.columns.tolist()[:3]

(9551, 21)


['Restaurant ID', 'Restaurant Name', 'Country Code']

In [3]:
# keep only India, then only Delhi NCR
india = df[df["Country Code"] == 1].copy()
india.columns = india.columns.str.lower().str.replace(" ", "_")

ncr = ["New Delhi", "Gurgaon", "Noida", "Faridabad", "Ghaziabad"]
data = india[india["city"].isin(ncr)].copy()

# drop columns that add no information
data = data.drop(columns=["country_code", "currency", "switch_to_order_menu"])

# handle missing / invalid values
data["cuisines"] = data["cuisines"].fillna("Unknown")
data["average_cost_for_two"] = data["average_cost_for_two"].replace(0, np.nan)

# flag unrated restaurants instead of deleting them
data["is_rated"] = data["aggregate_rating"] > 0

# flag chains (name appears 5+ times)
counts = data["restaurant_name"].value_counts()
data["is_chain"] = data["restaurant_name"].map(counts) >= 5

print(data.shape)
print("Unrated share:", round(1 - data["is_rated"].mean(), 3))
print("Chain restaurants:", data["is_chain"].sum())
print("Missing cost:", data["average_cost_for_two"].isna().sum())


(7947, 20)
Unrated share: 0.269
Chain restaurants: 1288
Missing cost: 0


In [4]:
data.to_csv("../data/zomato_ncr_clean.csv", index=False)